# Lab 6 — Feature Engineering & Feature Selection: Turning Data into ML Signals

**Course:** Machine Learning — Undergraduate
**Dataset:** Olist Brazilian E-Commerce Dataset *(synthetic stand-in — see note below)*
**Duration:** 2 Hours
**Starting Point:** `data/processed/olist_orders_abt.csv` created in Lab 2
**Prerequisites:** Labs 1–4; basic Python, pandas, NumPy, and scikit-learn

> **Note on data:** The real Olist ABT was not available in this environment, so a
> synthetic dataset with the same schema, realistic distributions, and a realistic
> ~12% late-delivery class imbalance was generated to stand in for the Lab 2 output.
> Swap in the real `olist_orders_abt.csv` and every cell below will run unchanged.


## 1. Lab Overview

In Lab 2, we created an Analytical Base Table (ABT): one row represented one customer
order, with information collected and aggregated from the raw Olist data.

In this lab, we do **not** rebuild the ABT. Instead, we take the existing ABT and ask:

> Can we represent the same business information in a way that makes it more useful
> for a machine learning model?

This is the purpose of **feature engineering**. We will also learn **feature
selection**: deciding which available features should actually be given to a model.

| Lab 2 — ABT Construction | Lab 6 — Feature Engineering |
|---|---|
| Combines information from multiple raw tables | Transforms information already in the ABT |
| Defines the analytical grain | Improves the representation of that grain |
| Uses joins and aggregations | Creates derived and transformed features |
| Main question: *"What data belongs in one row?"* | Main question: *"How can this data become a better ML signal?"* |


## 2. Learning Objectives

By the end of this lab, students should be able to:

1. Explain the difference between ABT construction and feature engineering.
2. Create numerical features from existing numerical columns.
3. Create ratio and interaction features.
4. Create useful date/time features.
5. Transform skewed numerical features.
6. Create simple business/domain features.
7. Apply basic feature selection techniques.
8. Detect and avoid feature leakage while engineering features.
9. Save a final engineered feature dataset for later ML experiments.
10. Explain why more features do not necessarily mean a better model.


## 3. Business Scenario

Imagine an e-commerce company wants to predict whether an order will be delivered
late. The ABT already contains information about each order (`total_price`,
`total_freight`, `total_items`, `unique_products`, `unique_sellers`, `order_month`,
`order_day_of_week`, `order_hour`, ...), but raw values do not always express the
relationship that matters.

For example: `total_price = R$2,500`, `total_freight = R$500` →
`freight_ratio = 500 / 2500 = 0.20`, i.e. freight is 20% of order value.

**Feature engineering creates a more meaningful representation of existing
information.**


## 7. Part A — Load and Inspect the ABT

### Step 1 — Import libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)

### Step 2 — Load the ABT

In [2]:
input_path = Path("data/processed/olist_orders_abt.csv")
df = pd.read_csv(input_path)
print("Shape:", df.shape)
df.head()

Shape: (6000, 20)


     order_id  customer_id customer_unique_id  total_price  total_freight  \
0  ord_000000  cust_000000        uniq_000000       561.36         149.73   
1  ord_000001  cust_000001        uniq_000001       110.71          21.02   
2  ord_000002  cust_000002        uniq_000002       196.84           3.94   
3  ord_000003  cust_000003        uniq_000003        69.96          12.41   
4  ord_000004  cust_000004        uniq_000004       127.96          23.79   

   total_items  unique_products  unique_sellers  order_year  order_month  \
0            5                4             2.0        2018            9   
1            4                4             2.0        2018           11   
2            4                4             2.0        2017            4   
3            3                3             1.0        2017            2   
4            2                2             2.0        2017            2   

   order_day  order_day_of_week  order_hour  delivery_days  \
0         17      

In [3]:
df.dtypes

order_id                    str
customer_id                 str
customer_unique_id          str
total_price             float64
total_freight           float64
total_items               int64
unique_products           int64
unique_sellers          float64
order_year                int64
order_month               int64
order_day                 int64
order_day_of_week         int64
order_hour                int64
delivery_days             int64
delivery_delay_days       int64
review_score              int64
review_comment_count      int64
has_review_comment        int64
is_low_review             int64
is_late_delivery          int64
dtype: object

**Questions**

1. How many rows are present?
2. How many columns are present?
3. What does one row represent?
4. What is the target column?
5. Which columns are identifiers?
6. Which columns are numerical?
7. Which columns are categorical?

In [4]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Numerical columns:", df.select_dtypes(include=np.number).columns.tolist())
print("Non-numerical columns:", df.select_dtypes(exclude=np.number).columns.tolist())

Rows: 6000
Columns: 20
Numerical columns: ['total_price', 'total_freight', 'total_items', 'unique_products', 'unique_sellers', 'order_year', 'order_month', 'order_day', 'order_day_of_week', 'order_hour', 'delivery_days', 'delivery_delay_days', 'review_score', 'review_comment_count', 'has_review_comment', 'is_low_review', 'is_late_delivery']
Non-numerical columns: ['order_id', 'customer_id', 'customer_unique_id']


## 8. Part B — Separate Target and Unsafe Columns

For this exercise:

In [5]:
target = "is_late_delivery"

identifier_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

leakage_columns = [
    "delivery_days",
    "delivery_delay_days",
    "review_score",
    "review_comment_count",
    "has_review_comment",
    "is_low_review"
]

Remove columns that are present in the ABT:

In [6]:
columns_to_drop = [
    c for c in identifier_columns + leakage_columns
    if c in df.columns
]
feature_df = df.drop(columns=columns_to_drop)
print(feature_df.columns.tolist())

['total_price', 'total_freight', 'total_items', 'unique_products', 'unique_sellers', 'order_year', 'order_month', 'order_day', 'order_day_of_week', 'order_hour', 'is_late_delivery']


**Think:** Why are identifiers generally poor ML features?
An identifier identifies a record; it usually does not explain the underlying
behavior.

## 9. Part C — Numerical Feature Engineering

Feature engineering does not always mean complicated mathematics. Simple
relationships can be highly useful.

### 9.1 Average Item Price

`average_item_price = total_price / total_items`

In [7]:
if {"total_price", "total_items"}.issubset(df.columns):
    df["average_item_price"] = (
        df["total_price"] /
        df["total_items"].replace(0, np.nan)
    )

### 9.2 Freight Ratio\n\n`freight_ratio = total_freight / total_price`

In [8]:
if {"total_freight", "total_price"}.issubset(df.columns):
    df["freight_ratio"] = (
        df["total_freight"] /
        df["total_price"].replace(0, np.nan)
    )

### 9.3 Items per Seller\n\nA different view of order complexity.

In [9]:
if {"total_items", "unique_sellers"}.issubset(df.columns):
    df["items_per_seller"] = (
        df["total_items"] /
        df["unique_sellers"].replace(0, np.nan)
    )

### 9.4 Seller Diversity

In [10]:
if {"unique_sellers", "total_items"}.issubset(df.columns):
    df["seller_diversity"] = (
        df["unique_sellers"] /
        df["total_items"].replace(0, np.nan)
    )

## 10. Part D — Date and Time Feature Engineering

Dates often contain much more information than one raw date value.

### 10.1 Weekend Indicator

Convention: `Monday=0 ... Saturday=5, Sunday=6`

In [11]:
if "order_day_of_week" in df.columns:
    df["is_weekend"] = (
        df["order_day_of_week"] >= 5
    ).astype(int)

### 10.2 Business-Hour Indicator

In [12]:
if "order_hour" in df.columns:
    df["is_business_hour"] = (
        df["order_hour"].between(9, 18)
    ).astype(int)

### 10.3 Year-End Indicator

The threshold is an instructional example; real features should be justified by
domain knowledge.

In [13]:
if "order_month" in df.columns:
    df["is_year_end"] = (
        df["order_month"].isin([11, 12])
    ).astype(int)

## 11. Part E — Cyclical Features

Time variables are often cyclical. December and January are close in time even
though their numerical values are 12 and 1.

`month_sin = sin(2π · month / 12)`, `month_cos = cos(2π · month / 12)`
`hour_sin = sin(2π · hour / 24)`, `hour_cos = cos(2π · hour / 24)`

**Why two columns?** Sine and cosine together preserve the circular structure.

In [14]:
if "order_month" in df.columns:
    df["month_sin"] = np.sin(2 * np.pi * df["order_month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["order_month"] / 12)

if "order_hour" in df.columns:
    df["hour_sin"] = np.sin(2 * np.pi * df["order_hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["order_hour"] / 24)

## 12. Part F — Log Transformation

E-commerce monetary variables may be positively skewed. A common transformation is
`x' = log(1 + x)`.

**Important:** A log transformation is not automatically better. Inspect the
distribution and have a reason for using it.

In [15]:
if "total_price" in df.columns:
    df["log_total_price"] = np.log1p(df["total_price"].clip(lower=0))

if "total_freight" in df.columns:
    df["log_total_freight"] = np.log1p(df["total_freight"].clip(lower=0))

In [16]:
df[["total_price", "log_total_price", "total_freight", "log_total_freight"]].describe().T

                    count        mean         std       min        25%  \
total_price        6000.0  211.659017  210.340925  5.550000  79.890000   
log_total_price    6000.0    4.986358    0.876560  1.879465   4.393090   
total_freight      5872.0   31.682730   38.146943  0.250000   9.217500   
log_total_freight  5872.0    3.000200    1.004942  0.223144   2.324102   

                          50%         75%          max  
total_price        146.760000  264.422500  2875.700000  
log_total_price      4.995589    5.581323     7.964399  
total_freight       19.480000   39.105000   740.340000  
log_total_freight    3.019449    3.691501     6.608459  

## 13. Part G — Binning

A continuous variable can sometimes be converted into meaningful groups. This can
represent a nonlinear relationship such as: very low → low → medium → high → very
high value order. The bin boundaries here are only an instructional example.

In [17]:
if "total_price" in df.columns:
    df["price_band"] = pd.cut(
        df["total_price"],
        bins=[-np.inf, 100, 500, 1000, 5000, np.inf],
        labels=["very_low", "low", "medium", "high", "very_high"]
    )

df["price_band"].value_counts()

price_band
low          3499
very_low     2002
medium        432
high           67
very_high       0
Name: count, dtype: int64

## 14. Part H — Interaction Features

Sometimes two variables together provide a useful signal. Do not create hundreds of
random combinations — ask: *what relationship am I trying to capture?*

In [18]:
if {"total_items", "total_price"}.issubset(df.columns):
    df["items_x_price"] = (
        df["total_items"] * df["total_price"]
    )

## 15. Part I — Inspect Invalid Values

Ratios can create missing values when the denominator is zero. Do not blindly
delete rows — as in Lab 4, missing-value handling should be part of a reproducible
preprocessing pipeline.

In [19]:
engineered_columns = [
    "average_item_price",
    "freight_ratio",
    "items_per_seller",
    "seller_diversity"
]
available_engineered = [c for c in engineered_columns if c in df.columns]
df[available_engineered].isna().sum()

average_item_price      0
freight_ratio         128
items_per_seller       73
seller_diversity       73
dtype: int64

In [20]:
df[available_engineered].describe().T

                     count       mean        std       min        25%  \
average_item_price  6000.0  70.677197  57.086672  2.775000  33.865000   
freight_ratio       5872.0   0.151469   0.076619  0.019791   0.096019   
items_per_seller    5927.0   1.584613   0.739975  1.000000   1.000000   
seller_diversity    5927.0   0.759579   0.283137  0.333333   0.500000   

                          50%        75%         max  
average_item_price  54.850000  89.088250  718.925000  
freight_ratio        0.149433   0.203772    0.477038  
items_per_seller     1.000000   2.000000    3.000000  
seller_diversity     1.000000   1.000000    1.000000  

## 16. Part J — Inspect the Engineered Features

In [21]:
new_columns = [
    "average_item_price",
    "freight_ratio",
    "items_per_seller",
    "seller_diversity",
    "is_weekend",
    "is_business_hour",
    "is_year_end",
    "month_sin",
    "month_cos",
    "hour_sin",
    "hour_cos",
    "log_total_price",
    "log_total_freight",
    "price_band",
    "items_x_price"
]
new_columns = [c for c in new_columns if c in df.columns]
df[new_columns].head()

   average_item_price  freight_ratio  items_per_seller  seller_diversity  \
0            112.2720       0.266727               2.5          0.400000   
1             27.6775       0.189865               2.0          0.500000   
2             49.2100       0.020016               2.0          0.500000   
3             23.3200       0.177387               3.0          0.333333   
4             63.9800       0.185917               1.0          1.000000   

   is_weekend  is_business_hour  is_year_end  month_sin     month_cos  \
0           0                 0            0  -1.000000 -1.836970e-16   
1           0                 0            1  -0.500000  8.660254e-01   
2           0                 1            0   0.866025 -5.000000e-01   
3           0                 0            0   0.866025  5.000000e-01   
4           0                 0            0   0.866025  5.000000e-01   

   hour_sin  hour_cos  log_total_price  log_total_freight price_band  \
0  0.707107  0.707107         6.

## 17. Part K — Feature Selection

Feature engineering can create many candidate features, but **more features does
not automatically mean a better model.** Feature selection asks: *which features
should we keep?* We use three introductory approaches.

### 17.1 Variance-Based Selection

A feature with only one value provides no useful variation.

In [22]:
from sklearn.feature_selection import VarianceThreshold

numeric_df = df.select_dtypes(include=np.number).copy()
numeric_df = numeric_df.drop(columns=[target], errors="ignore")

selector = VarianceThreshold(threshold=0.0)
selector.fit(numeric_df.fillna(numeric_df.median()))
selected_numeric = numeric_df.columns[selector.get_support()]

print("Original:", len(numeric_df.columns))
print("After variance filtering:", len(selected_numeric))

Original: 30
After variance filtering: 30


## 18. Part L — Correlation-Based Selection

Highly correlated features can contain overlapping information.

In [23]:
corr = numeric_df.corr(numeric_only=True)

threshold = 0.90
upper = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

high_corr_pairs = []
for col in upper.columns:
    for row in upper.index:
        value = upper.loc[row, col]
        if pd.notna(value) and abs(value) > threshold:
            high_corr_pairs.append((row, col, value))

high_corr_pairs[:20]

[('total_items', 'unique_products', np.float64(0.9513600742517285)), ('delivery_days', 'delivery_delay_days', np.float64(0.9999999999999997)), ('items_per_seller', 'seller_diversity', np.float64(-0.9721159378358517)), ('total_price', 'items_x_price', np.float64(0.9233720631525156))]

**Discussion:** Suppose `total_price` and `log_total_price` are strongly
related. Do we necessarily need both? Not always — the answer depends on the model
and the experiment.

## 19. Part M — Mutual Information

Mutual information can help identify nonlinear relationships between a feature and
the target.

In [24]:
from sklearn.feature_selection import mutual_info_classif

mi_df = df.select_dtypes(include=np.number).copy()
mi_df = mi_df.drop(columns=[target], errors="ignore")
mi_df = mi_df.fillna(mi_df.median())

mi_scores = mutual_info_classif(mi_df, df[target], random_state=42)

mi_results = pd.DataFrame({
    "feature": mi_df.columns,
    "mutual_information": mi_scores
}).sort_values("mutual_information", ascending=False)

mi_results.head(15)

                 feature  mutual_information
12          review_score            0.242043
15         is_low_review            0.201232
11   delivery_delay_days            0.058065
10         delivery_days            0.054187
13  review_comment_count            0.038233
17         freight_ratio            0.032075
8      order_day_of_week            0.025043
20            is_weekend            0.022638
14    has_review_comment            0.021672
4         unique_sellers            0.016821
28     log_total_freight            0.014049
1          total_freight            0.013221
2            total_items            0.009631
29         items_x_price            0.008209
19      seller_diversity            0.008156

**Interpretation:** A higher score indicates stronger statistical
dependence between the feature and target. It does not prove causality or
guarantee usefulness in every model.

## 20. Part N — Model-Based Feature Importance

Tree-based models can provide feature importance.

In [25]:
from sklearn.ensemble import RandomForestClassifier

X = df.select_dtypes(include=np.number).drop(columns=[target], errors="ignore")
X = X.fillna(X.median())
y = df[target]

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
rf.fit(X, y)

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

importance.head(15)

                 feature  importance
12          review_score    0.285153
15         is_low_review    0.207765
10         delivery_days    0.082229
11   delivery_delay_days    0.059665
17         freight_ratio    0.049275
13  review_comment_count    0.036503
8      order_day_of_week    0.027153
1          total_freight    0.020852
20            is_weekend    0.020122
28     log_total_freight    0.019785
14    has_review_comment    0.019411
16    average_item_price    0.016067
4         unique_sellers    0.015285
29         items_x_price    0.015033
0            total_price    0.014696

**Important caution:** Feature importance is not causality. An important
feature means the fitted model found it useful for prediction under this
experiment. It does not mean the feature *causes* late delivery.

## 21. Part O — Leakage Check

Before accepting an engineered feature, ask:

1. Was the feature available at prediction time?
2. Does it directly or indirectly reveal the target?
3. Was information from the test set used to create it?
4. Was the feature created using the target?
5. Was a transformation fitted using the complete dataset before train-test
   splitting?

If future or test information was used, the process may be leaking information.


## 22. Feature Engineering vs Feature Selection

| Feature Engineering | Feature Selection |
|---|---|
| Creates or transforms features | Chooses features |
| Changes representation | Reduces representation |
| Can increase the number of features | Usually decreases the number |
| Uses domain knowledge and transformations | Uses statistical/model-based criteria |
| Example: `freight_ratio` | Example: remove redundant feature |
| Example: `month_sin` | Example: keep top MI features |

**Memory aid:** Engineering creates candidates. Selection chooses candidates.

## 23. Part P — Create a Final Feature Set

In [26]:
candidate_features = [
    "total_price",
    "total_freight",
    "total_items",
    "unique_products",
    "unique_sellers",
    "order_month",
    "order_day_of_week",
    "order_hour",
    "average_item_price",
    "freight_ratio",
    "items_per_seller",
    "seller_diversity",
    "is_weekend",
    "is_business_hour",
    "month_sin",
    "month_cos",
    "hour_sin",
    "hour_cos",
    "log_total_price",
    "log_total_freight",
    "items_x_price"
]

final_features = [c for c in candidate_features if c in df.columns]
final_df = df[final_features + [target]].copy()

print("Final shape:", final_df.shape)
final_df.head()

Final shape: (6000, 22)


   total_price  total_freight  total_items  unique_products  unique_sellers  \
0       561.36         149.73            5                4             2.0   
1       110.71          21.02            4                4             2.0   
2       196.84           3.94            4                4             2.0   
3        69.96          12.41            3                3             1.0   
4       127.96          23.79            2                2             2.0   

   order_month  order_day_of_week  order_hour  average_item_price  \
0            9                  4           3            112.2720   
1           11                  3           3             27.6775   
2            4                  4          16             49.2100   
3            2                  1           4             23.3200   
4            2                  3          22             63.9800   

   freight_ratio  items_per_seller  seller_diversity  is_weekend  \
0       0.266727               2.5        

## 24. Part Q — Save the Engineered Dataset

In [27]:
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "olist_orders_feature_engineered.csv"
final_df.to_csv(output_path, index=False)
print("Saved to:", output_path)

Saved to: data/processed/olist_orders_feature_engineered.csv


## 25. Mini Experiment — Did Feature Engineering Add Value?

Compare:

- **Experiment A — Baseline:** a small set of original features.
- **Experiment B — Engineered:** baseline features plus selected engineered
  features.

Same train/test split, evaluation metric, model, and random seed for both.
Feature engineering is an **empirical process, not a guarantee of improvement.**

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

baseline_features = [
    c for c in [
        "total_price", "total_freight", "total_items",
        "unique_products", "unique_sellers",
        "order_month", "order_day_of_week", "order_hour"
    ] if c in df.columns
]

engineered_features = [
    c for c in final_features if c not in baseline_features
]

def evaluate(feature_list, label):
    X = df[feature_list].fillna(df[feature_list].median())
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    print(f"{label:12s} | features={len(feature_list):2d} | "
          f"accuracy={acc:.3f} | f1={f1:.3f}")
    return acc, f1

acc_base, f1_base = evaluate(baseline_features, "Baseline")
acc_eng, f1_eng = evaluate(baseline_features + engineered_features, "Engineered")

experiment_summary = pd.DataFrame({
    "Experiment": ["Baseline", "Engineered"],
    "Num Features": [len(baseline_features), len(baseline_features) + len(engineered_features)],
    "Accuracy": [acc_base, acc_eng],
    "F1": [f1_base, f1_eng]
})
experiment_summary

Baseline     | features= 8 | accuracy=0.677 | f1=0.347
Engineered   | features=21 | accuracy=0.697 | f1=0.366


   Experiment  Num Features  Accuracy        F1
0    Baseline             8  0.676667  0.346801
1  Engineered            21  0.696667  0.365854

## 26. Suggested Student Deliverables

1. **Notebook:** `Lab06_Feature_Engineering_and_Selection.ipynb` (this file)
2. **Engineered dataset:** `data/processed/olist_orders_feature_engineered.csv`
3. **Short feature engineering report**

| Item | Student Response |
|---|---|
| Original number of features | *(fill in)* |
| Number of engineered features | *(fill in)* |
| Most useful engineered feature | *(fill in)* |
| Why was it created? | *(fill in)* |
| Features removed during selection | *(fill in)* |
| Why were they removed? | *(fill in)* |
| Leakage risks found | *(fill in)* |
| Baseline model result | *(fill in from Part 25 above)* |
| Engineered model result | *(fill in from Part 25 above)* |
| Final conclusion | *(fill in)* |


## 27. Challenge Exercise

Choose three original features and create at least one meaningful engineered
feature from each. For each new feature, document:

- Original feature(s)
- New feature
- Formula
- Business interpretation
- Why might it help ML?
- Potential leakage risk

**Example**

- Original feature(s): `total_freight`, `total_price`
- New feature: `freight_ratio`
- Formula: `total_freight / total_price`
- Business interpretation: Share of order value spent on freight.
- Why might it help ML? It normalizes freight cost across different order sizes.
- Potential leakage risk: Check that both values are available at prediction time.

*(Complete this exercise for two more feature pairs of your choice.)*

## 28. Reflection

**Before this lab:** I thought a feature was...

**After this lab:** I now understand that a feature is...

**Most useful engineered feature:** The feature I think could help the model most
is...

**Most important lesson:** The most important thing I learned about feature
engineering is...

## 29. Key Takeaways

1. **ABT is not feature engineering.** ABT construction brings together the
   information needed to represent an analytical entity.
2. **Feature engineering changes representation.** We transform existing
   information into potentially more useful ML signals.
3. **Feature selection controls complexity.** We do not want every available
   feature automatically entering the model.
4. **Domain knowledge matters.** Good features often come from understanding the
   business problem.
5. **Avoid leakage.** A feature that knows the future can make a model look
   excellent while making the real system useless.


## 30. Final Mental Model

```
DATA
  ↓
┌──────────────┐
│     ABT      │   "Understand it"
└──────────────┘
  ↓
┌──────────────┐
│ Data Quality │   "Prepare it"
└──────────────┘
  ↓
┌──────────────┐
│   Pipeline   │   "Make signals"
└──────────────┘
  ↓
┌──────────────┐
│   Feature    │   "Choose signals"
│ Engineering  │
└──────────────┘
  ↓
┌──────────────┐
│   Feature    │
│  Selection   │
└──────────────┘
  ↓
MACHINE LEARNING
```

- Lab 2 asks: *"What information should represent an order?"*
- This lab asks: *"How can we represent that information better for learning?"*
